# Full-Text Search Basics

## Overview
Full-text search (FTS) goes beyond `ILIKE '%keyword%'` by tokenising, stemming, and indexing document text for fast, relevant searching. PostgreSQL's FTS system uses two core types: **tsvector** (document representation) and **tsquery** (search query), connected by the **@@** match operator.

**Core concepts:**

| Object | Description |
|---|---|
| `tsvector` | Sorted list of lexemes (stemmed word roots) with position info — the indexed document |
| `tsquery` | Boolean/phrase query of lexemes — what you're searching for |
| `@@` | Match operator: `tsvector @@ tsquery → boolean` |
| `to_tsvector()` | Tokenise + stem + remove stopwords → tsvector |
| `to_tsquery()` | Parse query string → tsquery |
| `websearch_to_tsquery()` | User-friendly query parsing (minus, quotes, spaces-as-AND) |

**Why FTS over ILIKE:**
- `ILIKE '%diabetes%'` → seq scan, O(N), no stemming, no ranking
- `tsv @@ to_tsquery('diabetes')` → GIN index, O(log N), stems to match 'diabetic', ranked

**Dataset:** clinical notes, research articles, and medications for 5 patients — healthcare text rich with medical terminology, abbreviations, and numeric values.

---

In [1]:
import sqlite3, json

conn = sqlite3.connect(':memory:')
conn.row_factory = sqlite3.Row

conn.executescript("""
-- Clinical notes and patient records for FTS demos
CREATE TABLE patients (
    patient_id  TEXT PRIMARY KEY,
    full_name   TEXT NOT NULL,
    province    TEXT NOT NULL,
    dob         TEXT NOT NULL
);

CREATE TABLE clinical_notes (
    note_id     TEXT PRIMARY KEY,
    patient_id  TEXT NOT NULL REFERENCES patients(patient_id),
    note_date   TEXT NOT NULL,
    provider    TEXT NOT NULL,
    note_type   TEXT NOT NULL,  -- 'progress','discharge','consult','operative'
    content     TEXT NOT NULL
);

CREATE TABLE medications (
    med_id      TEXT PRIMARY KEY,
    patient_id  TEXT NOT NULL,
    drug_name   TEXT NOT NULL,
    indication  TEXT,
    notes       TEXT
);

CREATE TABLE research_articles (
    article_id  TEXT PRIMARY KEY,
    title       TEXT NOT NULL,
    abstract    TEXT NOT NULL,
    keywords    TEXT,
    published   TEXT NOT NULL
);

-- FTS5 virtual tables (SQLite equivalent of PostgreSQL tsvector index)
CREATE VIRTUAL TABLE notes_fts USING fts5(
    note_id UNINDEXED,
    patient_id UNINDEXED,
    note_type UNINDEXED,
    content,
    tokenize='porter ascii'   -- porter stemmer (closest to pg's english dictionary)
);

CREATE VIRTUAL TABLE articles_fts USING fts5(
    article_id UNINDEXED,
    title,
    abstract,
    keywords,
    tokenize='porter ascii'
);
""")

patients = [
    ('P001','Alice Ngata',      'NB','1980-03-15'),
    ('P002','Bob Chen',         'ON','1972-07-22'),
    ('P003','Fatima Rashid',    'BC','1986-11-05'),
    ('P004','James MacLeod',    'NS','1963-01-30'),
    ('P005','Mei-Ling Tran',    'QC','1966-08-19'),
]
conn.executemany("INSERT INTO patients VALUES (?,?,?,?)", patients)

notes = [
    ('N001','P001','2023-10-01','Dr. Lee','progress',
     'Patient presents with poorly controlled hypertension. Blood pressure 148/92. '
     'Currently on Lisinopril 10mg once daily. Discussed medication adherence and sodium restriction. '
     'Patient reports occasional dizziness and dry cough, a known side effect of ACE inhibitors. '
     'Recommended dietary changes and increased physical activity. Follow-up in 4 weeks.'),
    ('N002','P001','2024-01-15','Dr. Lee','progress',
     'Follow-up for hypertension management. Blood pressure improved to 132/84. '
     'Patient adherent to Lisinopril. Dry cough persists. Considering switching to ARB if cough continues. '
     'HbA1c 7.2%, borderline elevated. Discussed diabetes prevention strategies. '
     'Lipid panel ordered. Continue current antihypertensive therapy.'),
    ('N003','P002','2024-01-10','Dr. Pham','discharge',
     'Patient admitted for acute exacerbation of Type 2 Diabetes. HbA1c 8.4% on admission. '
     'Adjusted Metformin to 1000mg BID and added Empagliflozin 10mg daily. '
     'Nutritional counselling provided. Patient educated on carbohydrate counting and glucose monitoring. '
     'Discharged in stable condition with follow-up appointment in 2 weeks.'),
    ('N004','P003','2023-08-20','Dr. Singh','consult',
     'Respiratory consultation for persistent asthma exacerbations. '
     'Patient reports frequent nocturnal wheezing and dyspnoea on exertion. '
     'Peak flow measurements show significant variability. Spirometry confirms obstructive pattern. '
     'Current inhaler technique reviewed and corrected. Prescribed Fluticasone-Salmeterol combination inhaler. '
     'Referral for pulmonary rehabilitation. Advised to avoid known allergens including dust mites and pet dander.'),
    ('N005','P004','2023-09-01','Dr. Lee','progress',
     'Routine diabetic review. HbA1c 7.8%, improved from last visit. '
     'eGFR 72 mL/min, stable kidney function. Patient reports no symptoms of hypoglycaemia. '
     'Foot examination normal. Retinal screening up to date. '
     'Continue current diabetes management. Annual flu vaccination administered. Blood pressure well controlled.'),
    ('N006','P005','2024-02-01','Dr. Pham','progress',
     'Complex case: Type 2 Diabetes with Hypertension and CKD Stage 3. '
     'HbA1c 9.1%, suboptimal glycaemic control. eGFR 38 mL/min, declining renal function. '
     'Metformin contraindicated due to renal impairment. Switched to Insulin glargine 20 units nocte. '
     'Potassium 5.8 mmol/L, hyperkalaemia noted. Dietary potassium restriction advised. '
     'Nephrology referral placed. Strict blood pressure control essential to slow CKD progression.'),
    ('N007','P002','2023-05-15','Dr. Pham','operative',
     'Pre-operative assessment for elective cholecystectomy. '
     'Medical history: Type 2 Diabetes, Hypertension, well controlled on Metformin and Lisinopril. '
     'Electrocardiogram normal. Chest X-ray clear. Blood glucose within acceptable range. '
     'Patient advised to withhold Metformin 48 hours prior to surgery due to contrast risk. '
     'Anaesthesia consultation arranged. Patient consented and cleared for surgery.'),
    ('N008','P001','2024-03-15','Dr. Singh','consult',
     'Cardiology consult for management of resistant hypertension. '
     'Despite optimal doses of Lisinopril and Amlodipine, blood pressure remains elevated at 158/96. '
     'Echocardiogram shows mild left ventricular hypertrophy. '
     'Added Spironolactone 25mg daily for additional blood pressure reduction. '
     'Stress test recommended to rule out ischaemic heart disease. '
     'Patient to monitor blood pressure at home twice daily and maintain log.'),
]
conn.executemany("INSERT INTO clinical_notes VALUES (?,?,?,?,?,?)", notes)

articles = [
    ('A001','Management of Type 2 Diabetes with Renal Impairment',
     'Type 2 diabetes mellitus complicated by chronic kidney disease requires careful medication selection. '
     'Metformin should be avoided when eGFR falls below 30 mL/min due to risk of lactic acidosis. '
     'SGLT2 inhibitors demonstrate renoprotective effects and cardiovascular benefits. '
     'Insulin therapy remains an option at all stages of renal impairment.',
     'diabetes,CKD,renal impairment,Metformin,SGLT2 inhibitor','2023-06-01'),
    ('A002','Hypertension Treatment Guidelines: ACE Inhibitors and ARBs',
     'Angiotensin-converting enzyme inhibitors and angiotensin receptor blockers are first-line '
     'antihypertensive agents. ACE inhibitors commonly cause dry cough due to bradykinin accumulation. '
     'ARBs are preferred in patients who cannot tolerate ACE inhibitor cough. '
     'Both drug classes provide renoprotection in diabetic nephropathy.',
     'hypertension,ACE inhibitor,ARB,Lisinopril,blood pressure','2022-11-15'),
    ('A003','Asthma Exacerbation Management in Primary Care',
     'Acute asthma exacerbations require rapid assessment of severity using peak expiratory flow and '
     'oxygen saturation. Short-acting beta-agonists remain the cornerstone of acute bronchodilation. '
     'Inhaled corticosteroids reduce airway inflammation and prevent future exacerbations. '
     'Patients with frequent exacerbations should be referred for specialist pulmonary assessment.',
     'asthma,exacerbation,bronchodilator,corticosteroid,peak flow','2023-03-20'),
    ('A004','Glycaemic Control Targets in Hospitalised Patients',
     'Inpatient hyperglycaemia is associated with increased morbidity, infection risk, and length of stay. '
     'Target blood glucose range of 6-10 mmol/L is recommended for most hospitalised patients. '
     'Insulin infusion protocols allow precise glycaemic management in critically ill patients. '
     'HbA1c measurement on admission provides information about pre-admission glycaemic control.',
     'glycaemic control,HbA1c,insulin,hospitalisation,hyperglycaemia','2024-01-08'),
    ('A005','Chronic Kidney Disease Progression and Blood Pressure Control',
     'Hypertension is both a cause and consequence of chronic kidney disease. '
     'Strict blood pressure control below 130/80 mmHg slows CKD progression significantly. '
     'Renin-angiotensin-aldosterone system blockade with ACE inhibitors or ARBs is recommended '
     'for patients with proteinuria. Regular monitoring of eGFR and potassium is essential.',
     'CKD,hypertension,blood pressure,ACE inhibitor,eGFR,proteinuria','2023-09-12'),
]
conn.executemany("INSERT INTO research_articles VALUES (?,?,?,?,?)", articles)

meds = [
    ('M001','P001','Lisinopril',   'Hypertension','10mg once daily'),
    ('M002','P001','Amlodipine',   'Hypertension','5mg once daily'),
    ('M003','P002','Metformin',    'Type 2 Diabetes','500mg BID'),
    ('M004','P002','Lisinopril',   'Hypertension','5mg once daily'),
    ('M005','P003','Salbutamol',   'Asthma','PRN inhaler'),
    ('M006','P005','Insulin glargine','Type 2 Diabetes','20 units nocte'),
    ('M007','P004','Metformin',    'Type 2 Diabetes','500mg daily'),
]
conn.executemany("INSERT INTO medications VALUES (?,?,?,?,?)", meds)

# Populate FTS5 indexes
conn.execute("INSERT INTO notes_fts SELECT note_id,patient_id,note_type,content FROM clinical_notes")
conn.execute("""
    INSERT INTO articles_fts
    SELECT article_id, title, abstract, COALESCE(keywords,'') FROM research_articles
""")
conn.commit()

print("FTS healthcare dataset loaded:")
for tbl in ['patients','clinical_notes','medications','research_articles']:
    n = conn.execute(f"SELECT COUNT(*) FROM {tbl}").fetchone()[0]
    print(f"  {tbl:<22s}: {n} rows")
print("  notes_fts (FTS5)      : indexed")
print("  articles_fts (FTS5)   : indexed")


FTS healthcare dataset loaded:
  patients              : 5 rows
  clinical_notes        : 8 rows
  medications           : 7 rows
  research_articles     : 5 rows
  notes_fts (FTS5)      : indexed
  articles_fts (FTS5)   : indexed


---
## tsvector: the document representation

In [2]:
print("=== tsvector: the document representation ===")
print()

print("A tsvector is a sorted list of lexemes (normalised word roots) with position info.")
print("PostgreSQL to_tsvector() tokenises, stems, and removes stop words.")
print()

pg_examples = [
    ("to_tsvector('english', 'Hypertension management with ACE inhibitors')",
     "'ace':4 'antihypertens':? 'inhibitor':5 'manag':2"),
    ("to_tsvector('english', 'Patient reports dry cough and dizziness')",
     "'cough':4 'dizziness':6 'dry':3 'patient':1 'report':2"),
    ("to_tsvector('english', 'HbA1c 7.2% borderline elevated diabetes')",
     "'7.2':2 'borderlin':3 'diabet':5 'elev':4 'hba1c':1"),
]
print("PostgreSQL to_tsvector() output (english dictionary):")
for sql, result in pg_examples:
    print(f"  {sql}")
    print(f"  → {result}")
    print()

print("What to_tsvector does:")
steps = [
    ("Tokenise",      "Split text into words and punctuation tokens"),
    ("Normalise",     "Lowercase all tokens"),
    ("Stem",          "Reduce to root form: 'inhibitors' → 'inhibitor', 'managing' → 'manag'"),
    ("Remove stops",  "Drop common words: 'and', 'the', 'with', 'a' are discarded"),
    ("Record positions","Each lexeme gets position numbers for phrase search"),
]
for step, desc in steps:
    print(f"  {step:<16s}  {desc}")

print()
print("SQLite FTS5 equivalent (porter stemmer):")
# Live demo with FTS5
rows = conn.execute("""
    SELECT content FROM notes_fts
    WHERE notes_fts MATCH 'hypertension'
    LIMIT 3
""").fetchall()
print(f"  MATCH 'hypertension' → {len(rows)} notes found")
for r in rows:
    snippet = r['content'][:100].replace('\n', ' ')
    print(f"    ...{snippet}...")

print()
print("PostgreSQL DDL for tsvector column:")
print("""
  -- Option 1: generated column (auto-updated)
  ALTER TABLE clinical_notes
      ADD COLUMN tsv tsvector
      GENERATED ALWAYS AS (
          to_tsvector('english', content)
      ) STORED;

  -- Option 2: manual column + trigger
  ALTER TABLE clinical_notes ADD COLUMN tsv tsvector;
  UPDATE clinical_notes SET tsv = to_tsvector('english', content);

  -- Then index it:
  CREATE INDEX idx_notes_tsv ON clinical_notes USING GIN (tsv);
""")


=== tsvector: the document representation ===

A tsvector is a sorted list of lexemes (normalised word roots) with position info.
PostgreSQL to_tsvector() tokenises, stems, and removes stop words.

PostgreSQL to_tsvector() output (english dictionary):
  to_tsvector('english', 'Hypertension management with ACE inhibitors')
  → 'ace':4 'antihypertens':? 'inhibitor':5 'manag':2

  to_tsvector('english', 'Patient reports dry cough and dizziness')
  → 'cough':4 'dizziness':6 'dry':3 'patient':1 'report':2

  to_tsvector('english', 'HbA1c 7.2% borderline elevated diabetes')
  → '7.2':2 'borderlin':3 'diabet':5 'elev':4 'hba1c':1

What to_tsvector does:
  Tokenise          Split text into words and punctuation tokens
  Normalise         Lowercase all tokens
  Stem              Reduce to root form: 'inhibitors' → 'inhibitor', 'managing' → 'manag'
  Remove stops      Drop common words: 'and', 'the', 'with', 'a' are discarded
  Record positions  Each lexeme gets position numbers for phrase searc

---
## tsquery: the search query

In [3]:
print("=== tsquery: the search query ===")
print()

print("tsquery types and operators:")
operators = [
    ("to_tsquery('english', 'hypertension')",
     "Single term: matches 'hypertension', 'hypertensive', 'antihypertensive'"),
    ("to_tsquery('english', 'diabetes & insulin')",
     "AND: both terms must appear in the document"),
    ("to_tsquery('english', 'diabetes | hypertension')",
     "OR: either term matches"),
    ("to_tsquery('english', 'diabetes & !insulin')",
     "NOT: diabetes present, insulin absent"),
    ("to_tsquery('english', 'blood <-> pressure')",
     "Phrase: blood immediately followed by pressure (positional)"),
    ("to_tsquery('english', 'blood <2> pressure')",
     "Near: blood within 2 positions of pressure"),
    ("plainto_tsquery('english', 'blood pressure control')",
     "Plain: spaces treated as AND (no operators needed)"),
    ("phraseto_tsquery('english', 'blood pressure control')",
     "Phrase: exact phrase match"),
    ("websearch_to_tsquery('english', 'diabetes -insulin \"blood sugar\"')",
     "Web syntax: minus=NOT, quotes=phrase, spaces=AND"),
]
print(f"  {'Function / operator':<48s}  Meaning")
print("  " + "-"*85)
for fn, desc in operators:
    print(f"  {fn:<48s}  {desc}")

print()
print("The @@ match operator:")
print("  tsvector @@ tsquery → boolean")
print()
print("  SELECT note_id FROM clinical_notes")
print("  WHERE  to_tsvector('english', content)")
print("         @@ to_tsquery('english', 'hypertension & \"ACE inhibitor\"');")

print()
# Live SQLite FTS5 equivalents
query_demos = [
    ("hypertension AND diabetes",         "hypertension diabetes"),
    ("HbA1c OR blood pressure",           "HbA1c OR \"blood pressure\""),
    ("diabetes NOT insulin",              "diabetes NOT insulin"),
    ("exact phrase: blood pressure",      '"blood pressure"'),
]
print("SQLite FTS5 MATCH equivalents (live):")
for label, query in query_demos:
    try:
        n = conn.execute(
            f"SELECT COUNT(*) FROM notes_fts WHERE notes_fts MATCH ?", (query,)
        ).fetchone()[0]
        print(f"  {label:<38s}  → {n} notes matched")
    except Exception as e:
        print(f"  {label:<38s}  → error: {e}")


=== tsquery: the search query ===

tsquery types and operators:
  Function / operator                               Meaning
  -------------------------------------------------------------------------------------
  to_tsquery('english', 'hypertension')             Single term: matches 'hypertension', 'hypertensive', 'antihypertensive'
  to_tsquery('english', 'diabetes & insulin')       AND: both terms must appear in the document
  to_tsquery('english', 'diabetes | hypertension')  OR: either term matches
  to_tsquery('english', 'diabetes & !insulin')      NOT: diabetes present, insulin absent
  to_tsquery('english', 'blood <-> pressure')       Phrase: blood immediately followed by pressure (positional)
  to_tsquery('english', 'blood <2> pressure')       Near: blood within 2 positions of pressure
  plainto_tsquery('english', 'blood pressure control')  Plain: spaces treated as AND (no operators needed)
  phraseto_tsquery('english', 'blood pressure control')  Phrase: exact phrase match
  we

---
## @@ operator: running searches

In [4]:
print("=== @@ operator: running searches ===")
print()

# Basic searches
searches = [
    ("hypertension",                  "hypertension"),
    ("diabetes",                      "diabetes"),
    ("HbA1c",                         "HbA1c"),
    ("renal impairment",              '"renal impairment"'),
    ("diabetes AND renal",            "diabetes renal"),
    ("asthma OR wheezing",            "asthma OR wheezing"),
    ("hypertension NOT cough",        "hypertension NOT cough"),
    ("blood pressure",                '"blood pressure"'),
]

print("Searching clinical notes:")
print(f"  {'search term':<36s}  {'notes':>6s}  note IDs")
print("  " + "-"*60)
for label, query in searches:
    try:
        rows = conn.execute(
            "SELECT note_id FROM notes_fts WHERE notes_fts MATCH ?", (query,)
        ).fetchall()
        ids = ', '.join(r['note_id'] for r in rows)
        print(f"  {label:<36s}  {len(rows):>6d}  {ids}")
    except Exception as e:
        print(f"  {label:<36s}  error: {e}")

print()
# Search articles
print("Searching research articles:")
art_searches = [
    ("diabetes",              "diabetes"),
    ("ACE inhibitor",         '"ACE inhibitor"'),
    ("CKD renal",             "CKD renal"),
    ("blood pressure control",'"blood pressure"'),
]
print(f"  {'search':<36s}  {'articles':>8s}  titles")
print("  " + "-"*75)
for label, query in art_searches:
    rows = conn.execute(
        "SELECT article_id FROM articles_fts WHERE articles_fts MATCH ?", (query,)
    ).fetchall()
    ids = ', '.join(r['article_id'] for r in rows)
    print(f"  {label:<36s}  {len(rows):>8d}  {ids}")

print()
print("PostgreSQL @@ search pattern:")
print("""
  -- Basic term search:
  SELECT note_id, patient_id, note_date
  FROM   clinical_notes
  WHERE  to_tsvector('english', content) @@ to_tsquery('english', 'hypertension')
  ORDER  BY note_date DESC;

  -- Multi-term AND:
  SELECT note_id FROM clinical_notes
  WHERE  to_tsvector('english', content)
         @@ to_tsquery('english', 'diabetes & ''renal impairment''');

  -- websearch_to_tsquery (most user-friendly):
  SELECT note_id FROM clinical_notes
  WHERE  to_tsvector('english', content)
         @@ websearch_to_tsquery('english', 'hypertension -cough "blood pressure"');
""")


=== @@ operator: running searches ===

Searching clinical notes:
  search term                            notes  note IDs
  ------------------------------------------------------------
  hypertension                               5  N001, N002, N006, N007, N008
  diabetes                                   5  N002, N003, N005, N006, N007
  HbA1c                                      4  N002, N003, N005, N006
  renal impairment                           1  N006
  diabetes AND renal                         1  N006
  asthma OR wheezing                         1  N004
  hypertension NOT cough                     3  N006, N007, N008
  blood pressure                             5  N001, N002, N005, N006, N008

Searching research articles:
  search                                articles  titles
  ---------------------------------------------------------------------------
  diabetes                                     2  A001, A002
  ACE inhibitor                                2  A002, A005
  

---
## Common Pitfalls

**1. Using ILIKE instead of FTS for text search**
`WHERE content ILIKE '%hypertension%'` performs a full sequential scan — no index, O(N) cost, no stemming, no ranking. `ILIKE '%..%'` (leading wildcard) cannot use any B-tree index. For any non-trivial text search, FTS with a GIN index is orders of magnitude faster and returns more relevant results (stemming matches 'hypertensive' and 'antihypertensive').

**2. Calling to_tsvector() without specifying a language config**
`to_tsvector(content)` uses the `default_text_search_config` session variable, which may differ between development and production environments. Always specify explicitly: `to_tsvector('english', content)`. The expression index must use the same config or it won't be used.

**3. to_tsquery() failing on user input with special characters**
`to_tsquery('english', user_input)` throws an error if `user_input` contains characters like `&`, `|`, `(`, `)`, or `:`. Never pass raw user input to `to_tsquery()`. Use `websearch_to_tsquery()` or `plainto_tsquery()` for user-supplied search strings — they parse safely and don't throw on unusual input.

**4. Stop words silently removing search terms**
`to_tsvector('english', 'the best treatment')` → `'best':2 'treatment':3` — 'the' is a stop word and is removed. A search for `to_tsquery('english', 'the')` returns zero results because the tsvector contains no 'the' lexeme. Use `to_tsquery('simple', 'the')` or switch to `plainto_tsquery` if users expect stop words to match.

**5. FTS index not used because expression doesn't match exactly**
A GIN index on `to_tsvector('english', content)` is only used when the WHERE clause uses exactly `to_tsvector('english', content) @@ ...`. `to_tsvector('simple', content) @@` or `to_tsvector(content) @@` will trigger a seq scan. Match the index expression character-for-character in queries.

**6. Not accounting for NULL in tsvector expressions**
`to_tsvector('english', NULL)` returns NULL, not an empty tsvector. A stored tsvector column built from nullable columns should use COALESCE: `to_tsvector('english', COALESCE(content, ''))`. Otherwise NULLs are excluded from the index and those rows never match any search.


---
*sql_methods_library - Samantha McGarrigle*